In [0]:
-- Fetch the qdp_accounts_all document data product created by 01. Quantexa Processing of Raw Data in Bronze

-- 01-1. Create a point in time run_timestamp for all data inserts
DROP TEMPORARY VARIABLE IF EXISTS run_timestamp;

DECLARE run_timestamp TIMESTAMP DEFAULT current_timestamp();

SELECT run_timestamp;


-- 01-2. Create default start date for all data inserts

DROP TEMPORARY VARIABLE IF EXISTS default_start_date;

DECLARE default_start_date STRING;

SET  VAR default_start_date = '01-01-1900';

SELECT default_start_date;


-- 01-3. Create default end date for all data inserts
DROP TEMPORARY VARIABLE IF EXISTS default_end_date;

DECLARE default_end_date STRING;

SET  VAR default_end_date = '31-12-2999';

SELECT default_end_date;

-- 2. Remove all Account records for the Tennant

DELETE FROM demo_banking_silver.qdp_accounts_all.sat_account s
WHERE s.account_id IN (
  SELECT DISTINCT account_id
  FROM demo_banking_silver.qdp_accounts_all.hub_account
  WHERE tennant_id = :p_tennant_id)
;

DELETE FROM demo_banking_silver.qdp_accounts_all.hub_account WHERE tennant_id = :p_tennant_id;

-- 3. Assign the correct address_id or leave as defaulted zero value if does not exist
CREATE OR REPLACE TEMP VIEW accounts_with_correct_types_addresses AS
SELECT 
    q.*,
    COALESCE(haddr.address_id, 0) AS address_id
FROM demo_banking_silver.qdp_from_quantexa_banking_supply_chain.qdp_accounts_all q
LEFT JOIN demo_banking_silver.qdp_locations_all.hub_address haddr
ON q.address_entity_id = haddr.address_entity_id
;

SELECT * from accounts_with_correct_types_addresses;



-- 4. Assign the correct organisation_id or leave as defaulted zero value if does not exist

CREATE OR REPLACE TEMP VIEW accounts_with_correct_types_addresses_orgs AS
SELECT 
    q.*,
    COALESCE(horg.organisation_id, 0) AS organisation_id
FROM accounts_with_correct_types_addresses q
LEFT JOIN demo_banking_silver.qdp_organisations_all.hub_organisation horg
ON q.organisation_entity_id = horg.organisation_entity_id
;


SELECT * from accounts_with_correct_types_addresses_orgs;


-- 5. Assign the correct individual_id or leave as defaulted zero value if does not exist

CREATE OR REPLACE TEMP VIEW accounts_with_correct_types_addresses_orgs_indv AS
SELECT 
    q.*,
    COALESCE(hind.individual_id, 0) AS individual_id
FROM accounts_with_correct_types_addresses_orgs q
LEFT JOIN demo_banking_silver.qdp_individuals_all.hub_individual hind
ON q.individual_entity_id = hind.individual_entity_id
;


SELECT * from accounts_with_correct_types_addresses_orgs_indv;


-- 6. Assign the correct organisation_customer_id or leave as defaulted zero value if does not exist

CREATE OR REPLACE TEMP VIEW accounts_with_correct_types_addresses_orgs_indv_orgcust AS
SELECT 
    q.*,
    COALESCE(horgcust.organisation_customer_id, 0) AS organisation_customer_id
FROM accounts_with_correct_types_addresses_orgs_indv q
LEFT JOIN demo_banking_silver.qdp_organisation_customers.hub_organisation_customers horgcust
ON q.organisation_customer_entity_id = horgcust.organisation_customer_entity_id
;


SELECT * from accounts_with_correct_types_addresses_orgs_indv_orgcust;



-- 7. Assign the correct organisation_individual_id or leave as defaulted zero value if does not exist

CREATE OR REPLACE TEMP VIEW accounts_with_correct_types_addresses_orgs_indv_orgcust_indcust AS
SELECT 
    q.*,
    COALESCE(hindcust.individual_customer_id, 0) AS individual_customer_id
FROM accounts_with_correct_types_addresses_orgs_indv_orgcust q
LEFT JOIN demo_banking_silver.qdp_individual_customers.hub_individual_customers hindcust
ON q.individual_customer_entity_id = hindcust.customer_entity_id
;


SELECT * from accounts_with_correct_types_addresses_orgs_indv_orgcust_indcust;



-- 8. Create the hub_bank_account records
INSERT INTO demo_banking_silver.qdp_accounts_all.hub_account
( account_entity_id, 
  tennant_id,
  individual_customer_entity_id,
  individual_entity_id,
  organisation_customer_entity_id,
  organisation_entity_id,
  address_entity_id,
  product_entity_id, 
  period_start, 
  period_end)
SELECT 
  account_entity_id,
  tennant_id, 
  individual_customer_entity_id,
  individual_entity_id,
  organisation_customer_entity_id,
  organisation_entity_id,
  address_entity_id,
  product_entity_id, 
  period_start, 
  period_end
FROM accounts_with_correct_types_addresses_orgs_indv_orgcust_indcust;

SELECT * FROM demo_banking_silver.qdp_accounts_all.hub_account;




-- 9. Insert into sat_bank_account for Individual Customers
INSERT INTO demo_banking_silver.qdp_accounts_all.sat_account
    (account_id, 
     load_datetime, 
     record_source_id, 
     individual_customer_entity_id,  	
     individual_customer_id,  	
     individual_id,  	
     organisation_customer_entity_id,  	
     organisation_customer_id,
     organisation_id,  	
     address_entity_id,
     address_id,  	
     product_entity_id,
     product_id,  	
     sort_code,
     account_number,
     account_name,
     iban,
     swiftbic,
	   data_source_code,
	   data_source_concept_id,
	   data_source_raw_code,
	   data_source_raw_concept_id,
	   type_code,
	   type_concept_id,
	   type_raw_code,
	   type_raw_concept_id,
     balance,
     overdraft_limit,
     loan_original_amount,
     loan_orininal_maturity_date,
     servicer_identification,
     servicer_scheme_name,    
	   currency_code,
	   currency_concept_id,
	   currency_raw_code,
	   currency_raw_concept_id,
	   branch_code,
	   branch_concept_id,
	   branch_raw_code,
	   branch_raw_concept_id,
     opened_datetime,
     closed_datetime,
	   status_code,
	   status_concept_id,
	   status_raw_code,
	   status_raw_concept_id,
     risk_rating,
	   risk_rating_code,
	   risk_rating_concept_id,
	   risk_rating_raw_code,
	   risk_rating_raw_concept_id,
	   country_code,
	   country_concept_id,
	   country_raw_code,
	   country_raw_concept_id,
     is_account_alarm,
     is_bank_account,
     is_business_account,
     is_customer_acccount,
     is_counterparty_account,
     is_frozed,
     is_closed,
     is_open,
     is_excluded_from_monitoring,
     period_start,
     period_end
    )
SELECT
  h.account_id AS account_id,
  run_timestamp AS load_datetime,
  try_cast(0 AS BIGINT) AS record_source_id,
  q.individual_customer_entity_id,  	
  q.individual_customer_id AS individual_customer_id,
  q.individual_id,
  q.organisation_customer_entity_id,  	
  q.organisation_customer_id AS organisation_customer_id,
  q.organisation_id,  	
  q.address_entity_id,
  q.address_id AS address_id,
  q.product_entity_id,
--  q.product_id,
0,  	
  q.sort_code,
  q.account_number,
  q.account_name,
  q.iban,
  q.swiftbic,
  q.data_source_code,
  q.data_source_concept_id,
  q.data_source_raw_code,
  q.data_source_raw_concept_id,
  q.type_code,
  q.type_concept_id,
  q.type_raw_code,
  q.type_raw_concept_id,
  q.balance,
  q.overdraft_limit,
  q.loan_original_amount,
  q.loan_orininal_maturity_date,
  q.servicer_identification,
  q.servicer_scheme_name,    
  q.currency_code,
  q.currency_concept_id,
  q.currency_raw_code,
  q.currency_raw_concept_id,
  q.branch_code,
  q.branch_concept_id,
  q.branch_raw_code,
  q.branch_raw_concept_id,
  q.opened_datetime,
  q.closed_datetime,
  q.status_code,
  q.status_concept_id,
  q.status_raw_code,
  q.status_raw_concept_id,
  q.risk_rating,
  q.risk_rating_code,
  q.risk_rating_concept_id,
  q.risk_rating_raw_code,
  q.risk_rating_raw_concept_id,
  q.country_code,
  q.country_concept_id,
  q.country_raw_code,
  q.country_raw_concept_id,
  q.is_account_alarm,
  q.is_bank_account,
  q.is_business_account,
  q.is_customer_acccount,
  q.is_counterparty_account,
  q.is_frozed,
  q.is_closed,
  q.is_open,
  q.is_excluded_from_monitoring,
  q.period_start AS period_start,
  q.period_end AS period_end

FROM accounts_with_correct_types_addresses_orgs_indv_orgcust_indcust q
JOIN demo_banking_silver.qdp_accounts_all.hub_account h
  ON h.account_entity_id = q.account_entity_id
;
  
SELECT * FROM demo_banking_silver.qdp_accounts_all.sat_account;



 
